# 🚀 Semantic Search Application

**Day 5 — Notebook 4 of 4 | Estimated Time: 30 minutes**

---

## What You'll Learn
- Build a complete end-to-end semantic search system over a document corpus
- Combine semantic search with Gemini to answer questions (a preview of RAG)
- Evaluate search quality: is the right document being retrieved?
- Apply embeddings to classification and recommendation tasks

---

## Setup

In [ ]:
%pip install -U -q "google-genai>=1.0.0" numpy

In [ ]:
import sys, os
import numpy as np

repo_root = os.path.abspath(os.path.join(os.getcwd(), "../../.."))
if repo_root not in sys.path:
    sys.path.append(repo_root)

from helpers.auth import get_api_key
from google import genai
from google.genai import types
from IPython.display import Markdown

client = genai.Client(api_key=get_api_key())
EMBEDDING_MODEL = "text-embedding-004"
GEN_MODEL = "gemini-2.5-flash"
print("✅ Ready!")

---

## 1. The Document Corpus

We'll build a searchable knowledge base for a fictional **tech company's internal FAQ**. This is a realistic use case — teams search internal docs all the time.

In [ ]:
# Internal company knowledge base
DOCUMENTS = [
    {
        "id": "hr-001",
        "title": "Annual Leave Policy",
        "text": "Full-time employees are entitled to 25 days of annual leave per year. Leave must be approved by your line manager at least 2 weeks in advance. Up to 5 days can be carried over to the next year. Unused leave beyond 5 days is forfeited at year end.",
        "category": "HR",
    },
    {
        "id": "hr-002",
        "title": "Remote Work Policy",
        "text": "Employees may work remotely up to 3 days per week. A minimum of 2 days in the office is required. Remote work requests for full weeks must be approved by your manager. Home office equipment allowance is £500 per year.",
        "category": "HR",
    },
    {
        "id": "hr-003",
        "title": "Parental Leave",
        "text": "Primary caregivers receive 26 weeks of fully paid parental leave. Secondary caregivers receive 4 weeks of fully paid leave. Parental leave can start up to 2 weeks before the expected due date. Additional unpaid leave of up to 13 weeks is available.",
        "category": "HR",
    },
    {
        "id": "it-001",
        "title": "VPN Access",
        "text": "To access company systems remotely, install the GlobalProtect VPN client. Use your company email and password to authenticate. VPN access is required for all internal tools including Jira, Confluence, and the internal code repositories. Contact IT helpdesk for setup issues.",
        "category": "IT",
    },
    {
        "id": "it-002",
        "title": "Password Policy",
        "text": "Passwords must be at least 12 characters and include uppercase, lowercase, numbers, and symbols. Passwords expire every 90 days. Password reuse is blocked for the last 10 passwords. Multi-factor authentication (MFA) is mandatory for all accounts.",
        "category": "IT",
    },
    {
        "id": "it-003",
        "title": "Software Request Process",
        "text": "To request new software, submit a ticket via the IT portal at it.company.com. Include the software name, business justification, and cost. Requests under £100/year are approved automatically. Requests over £100/year require manager and finance sign-off. Processing time is 3-5 business days.",
        "category": "IT",
    },
    {
        "id": "fin-001",
        "title": "Expense Reimbursement",
        "text": "Submit expense claims via the Expenses app within 30 days of the expense. All claims require receipts. Meals are reimbursed up to £40 per day when travelling for work. UK domestic travel is reimbursed at standard rail class. Taxis require prior approval for amounts over £20.",
        "category": "Finance",
    },
    {
        "id": "fin-002",
        "title": "Company Credit Cards",
        "text": "Company credit cards are available for employees who regularly incur business expenses. Applications must be approved by your department head. Monthly spend limits are set per role. Personal use of company credit cards is strictly prohibited and may result in disciplinary action.",
        "category": "Finance",
    },
    {
        "id": "eng-001",
        "title": "Code Review Process",
        "text": "All code changes require at least 2 approvals before merging. One approval must be from a senior engineer or tech lead. PRs should be kept under 400 lines of change where possible. Automated tests must pass before requesting review. Reviews should be completed within 1 business day.",
        "category": "Engineering",
    },
    {
        "id": "eng-002",
        "title": "On-Call Rotation",
        "text": "Engineers join the on-call rotation after 6 months. On-call shifts are one week long. P1 incidents must be acknowledged within 15 minutes. Engineers receive a £200 on-call allowance per week. Incidents outside working hours can be escalated if not resolved within 30 minutes.",
        "category": "Engineering",
    },
]

print(f"Corpus size: {len(DOCUMENTS)} documents")
print("Categories:", set(d["category"] for d in DOCUMENTS))

---

## 2. Build the Search Index

In [ ]:
# Embed the full corpus
print("Building search index...")

texts = [doc["text"] for doc in DOCUMENTS]

result = client.models.embed_content(
    model=EMBEDDING_MODEL,
    contents=texts,
    config=types.EmbedContentConfig(task_type="RETRIEVAL_DOCUMENT"),
)

doc_vectors = np.array([e.values for e in result.embeddings])

print(f"✅ Index built: {len(DOCUMENTS)} documents × {doc_vectors.shape[1]} dimensions")


def semantic_search(query: str, top_k: int = 3, category: str = None) -> list[dict]:
    """Search the company knowledge base."""
    # Embed the query
    q_result = client.models.embed_content(
        model=EMBEDDING_MODEL,
        contents=query,
        config=types.EmbedContentConfig(task_type="RETRIEVAL_QUERY"),
    )
    q_vec = np.array(q_result.embeddings[0].values)
    
    # Score all documents
    scores = doc_vectors @ q_vec
    
    # Build result list with scores
    ranked = sorted(
        [{**doc, "score": float(score)} for doc, score in zip(DOCUMENTS, scores)],
        key=lambda x: x["score"],
        reverse=True,
    )
    
    # Optional category filter
    if category:
        ranked = [r for r in ranked if r["category"].lower() == category.lower()]
    
    return ranked[:top_k]

In [ ]:
# Test basic search
queries = [
    "How many days off do I get per year?",
    "I'm having trouble connecting to internal tools from home.",
    "How do I get reimbursed for a work dinner?",
    "What happens when there's a production outage?",
]

for q in queries:
    results = semantic_search(q, top_k=2)
    print(f"🔍 '{q}'")
    for r in results:
        print(f"   [{r['score']:.3f}] ({r['category']}) {r['title']}: {r['text'][:80]}...")
    print()

---

## 3. Search + Generate: A Preview of RAG

The real power of semantic search is combining it with a generative model. Retrieve the relevant context, then let the LLM answer in natural language. This is the core idea behind **RAG (Retrieval-Augmented Generation)**, which we'll cover fully in Day 7.

In [ ]:
def answer_question(question: str, top_k: int = 3) -> str:
    """
    1. Retrieve the most relevant documents via semantic search
    2. Feed them to Gemini as context
    3. Generate a grounded answer
    """
    # Step 1: Retrieve relevant documents
    results = semantic_search(question, top_k=top_k)
    
    if not results or results[0]["score"] < 0.5:
        return "I couldn't find relevant information in the knowledge base to answer this question."
    
    # Step 2: Format context
    context_parts = []
    for r in results:
        context_parts.append(f"[{r['title']} — {r['category']}]\n{r['text']}")
    context = "\n\n".join(context_parts)
    
    # Step 3: Generate answer
    response = client.models.generate_content(
        model=GEN_MODEL,
        contents=f"""Answer the employee's question using ONLY the company policies provided below.
Be concise and direct. If the answer is not covered, say so.

<company_policies>
{context}
</company_policies>

Employee question: {question}

Answer:""",
        config=types.GenerateContentConfig(temperature=0.1),
    )
    
    return response.text


# Ask the company knowledge base questions
questions = [
    "Can I work from home every day?",
    "My PR has been waiting 3 days — is that normal?",
    "How long does it take to get new software approved?",
    "What is the parental leave allowance?",
]

for q in questions:
    print(f"❓ {q}")
    answer = answer_question(q)
    print(f"🤖 {answer.strip()}")
    print("-" * 70)

---

## 4. Embedding-Based Text Classification

Embeddings can power **zero-shot classification** without training a classifier. Just embed the text and the category labels, then find the closest label.

In [ ]:
# Zero-shot classification using embedding similarity
category_descriptions = {
    "HR": "Human resources, employee policies, leave, benefits, and people management",
    "IT": "Technology, software, hardware, security, systems access, and IT support",
    "Finance": "Expenses, payments, budgets, reimbursements, and financial policies",
    "Engineering": "Software development, code reviews, deployments, and engineering processes",
}

# Embed category descriptions
cat_result = client.models.embed_content(
    model=EMBEDDING_MODEL,
    contents=list(category_descriptions.values()),
)
cat_vectors = np.array([e.values for e in cat_result.embeddings])
cat_names = list(category_descriptions.keys())


def classify_question(question: str) -> tuple[str, float]:
    """Classify a question into one of the predefined categories."""
    q_result = client.models.embed_content(
        model=EMBEDDING_MODEL,
        contents=question,
        config=types.EmbedContentConfig(task_type="RETRIEVAL_QUERY"),
    )
    q_vec = np.array(q_result.embeddings[0].values)
    
    scores = cat_vectors @ q_vec
    best_idx = np.argmax(scores)
    return cat_names[best_idx], float(scores[best_idx])


test_questions = [
    "How do I reset my password?",
    "Can I carry over unused holidays?",
    "What receipts do I need for expense claims?",
    "How many reviewers do I need for a merge?",
    "Is there a bring-your-own-device policy?",
]

for q in test_questions:
    category, score = classify_question(q)
    print(f"  {category:12s} [{score:.3f}]  {q}")

---

## 5. Evaluating Search Quality

Build a simple evaluation set to measure how often your search retrieves the right document.

In [ ]:
# Golden evaluation set: (query, expected_doc_id)
eval_set = [
    ("How many days annual leave do I have?", "hr-001"),
    ("Can I work from home full time?", "hr-002"),
    ("How do I connect to the VPN?", "it-001"),
    ("What are the password requirements?", "it-002"),
    ("How do I claim back work expenses?", "fin-001"),
    ("I need a new software tool for my team.", "it-003"),
    ("What is the maternity leave policy?", "hr-003"),
    ("How many code review approvals are needed?", "eng-001"),
]

correct_at_1 = 0
correct_at_3 = 0

print("Evaluation Results:\n")
for query, expected_id in eval_set:
    results = semantic_search(query, top_k=3)
    result_ids = [r["id"] for r in results]
    
    hit_at_1 = result_ids[0] == expected_id if result_ids else False
    hit_at_3 = expected_id in result_ids
    
    correct_at_1 += hit_at_1
    correct_at_3 += hit_at_3
    
    status = "✅" if hit_at_1 else ("⚠️" if hit_at_3 else "❌")
    print(f"  {status} Expected: {expected_id:8s}  Got: {result_ids[0]:8s}  Query: {query[:50]}")

print(f"\n📊 Precision@1: {correct_at_1}/{len(eval_set)} = {correct_at_1/len(eval_set):.0%}")
print(f"📊 Recall@3:    {correct_at_3}/{len(eval_set)} = {correct_at_3/len(eval_set):.0%}")

---

## 6. Embedding-Based Recommendation

"More like this" recommendations — find documents similar to a given document.

In [ ]:
def find_similar_docs(doc_id: str, top_k: int = 3) -> list[dict]:
    """Find documents most similar to a given document (excluding itself)."""
    # Find the document index
    doc_idx = next(i for i, d in enumerate(DOCUMENTS) if d["id"] == doc_id)
    doc_vec = doc_vectors[doc_idx]
    
    # Score all other documents
    scores = doc_vectors @ doc_vec
    
    ranked = sorted(
        [
            {**doc, "score": float(score)}
            for i, (doc, score) in enumerate(zip(DOCUMENTS, scores))
            if i != doc_idx  # Exclude the document itself
        ],
        key=lambda x: x["score"],
        reverse=True,
    )
    return ranked[:top_k]


# "You might also be interested in..."
print("📄 If you read 'Annual Leave Policy', you might also want:")
for r in find_similar_docs("hr-001"):
    print(f"   [{r['score']:.3f}] {r['title']} ({r['category']})")

print()
print("📄 If you read 'Code Review Process', you might also want:")
for r in find_similar_docs("eng-001"):
    print(f"   [{r['score']:.3f}] {r['title']} ({r['category']})")

---

## 🏋️ Exercise 1: Add New Documents and Search

Add your own policy documents to the knowledge base and search them.

In [ ]:
# Exercise 1: Extend the knowledge base
# TODO: Add 2-3 new documents to the corpus below
# Then re-build the index and test searches that find your new documents

new_documents = [
    # TODO: Add your own documents here
    # {
    #     "id": "your-id",
    #     "title": "Your Title",
    #     "text": "Your document text here...",
    #     "category": "HR" | "IT" | "Finance" | "Engineering",
    # },
]

# TODO:
# 1. Embed new_documents
# 2. Append to DOCUMENTS and doc_vectors (use np.vstack)
# 3. Test a query that should retrieve your new document

---

## 🏋️ Exercise 2: Build a Complete Q&A Bot

Combine everything from this notebook into a complete interactive Q&A loop.

In [ ]:
# Exercise 2: Full Q&A bot
# TODO: Build a complete Q&A bot that:
#   1. Takes a user question
#   2. Classifies which department (HR/IT/Finance/Engineering) it belongs to
#   3. Searches only that department's documents
#   4. Generates a grounded answer
#   5. Returns sources it used

def smart_qa_bot(question: str) -> dict:
    """
    Returns:
        {
            "answer": str,
            "category": str,
            "sources": list of document titles used
        }
    """
    # TODO: Implement
    # Step 1: classify the question to get a department
    # Step 2: search only that department using semantic_search with category filter
    # Step 3: generate an answer using the retrieved context
    # Step 4: return answer + metadata
    pass


# Test it
# result = smart_qa_bot("What is the process for getting new software at work?")
# print(f"Category: {result['category']}")
# print(f"Sources: {result['sources']}")
# print(f"Answer: {result['answer']}")

---

## What You've Built

Over these 4 notebooks, you've gone from "what is an embedding" to a **working semantic search application** that:

1. Embeds a corpus of documents using Gemini's embedding model
2. Retrieves relevant documents for any natural language query
3. Generates grounded answers using the retrieved context (a preview of RAG)
4. Classifies questions by topic using zero-shot embedding similarity
5. Measures retrieval quality with Precision@K

---

## Day 5 Complete! 🎉

You've learned:
- ✅ What embeddings are and how they capture meaning (Notebook 1)
- ✅ How to generate and manage embeddings with Gemini (Notebook 2)
- ✅ Cosine similarity and building a search index (Notebook 3)
- ✅ End-to-end semantic search application (Notebook 4)

**Next:** Day 6 — Vector Databases & Document Processing

---

## 📚 Further Reading

| Resource | Type | Link |
|----------|------|------|
| Gemini Embeddings | Docs | [ai.google.dev/gemini-api/docs/embeddings](https://ai.google.dev/gemini-api/docs/embeddings) |
| Semantic Search Explained | Blog | [huggingface.co/blog/getting-started-with-embeddings](https://huggingface.co/blog/getting-started-with-embeddings) |
| Retrieval-Augmented Generation | Preview | Coming in Day 7 |